In [ ]:
# Hosted D2L setup: fetch the exact helper module used to build this notebook.
from pathlib import Path
from urllib.request import urlretrieve
from importlib.metadata import PackageNotFoundError, version
import importlib.util, os, subprocess, sys

required = ['numpy', 'pandas', 'matplotlib', 'requests', 'scipy', 'pillow', 'regex', 'jax', 'jaxlib', 'flax', 'optax', 'orbax-checkpoint', 'tensorflow', 'protobuf', 'ml-dtypes']
imports = {'pillow': 'PIL', 'orbax-checkpoint': 'orbax', 'protobuf': 'google.protobuf', 'ml-dtypes': 'ml_dtypes'}
pinned = {'jax': ('0.10.2', 'jax==0.10.2', 'jax[cuda12]==0.10.2', 'exact'), 'jaxlib': ('0.10.2', 'jaxlib==0.10.2', 'jaxlib==0.10.2', 'exact'), 'flax': ('0.12.7', 'flax==0.12.7', 'flax==0.12.7', 'exact'), 'optax': ('0.2.8', 'optax==0.2.8', 'optax==0.2.8', 'exact'), 'orbax-checkpoint': ('0.12.0', 'orbax-checkpoint==0.12.0', 'orbax-checkpoint==0.12.0', 'exact')}
fallbacks = {'tensorflow': 'tensorflow==2.21.0', 'protobuf': 'protobuf==7.34.1', 'ml-dtypes': 'ml-dtypes==0.5.4'}
device = os.environ.get("D2L_HOSTED_DEVICE", "auto").lower()
if device not in ("auto", "cpu", "gpu"):
    raise ValueError(f"Invalid D2L_HOSTED_DEVICE={device!r}")
if device == "auto":
    try:
        gpu = (Path("/dev/nvidia0").exists() or
               subprocess.run(["nvidia-smi", "-L"], capture_output=True,
                              timeout=5).returncode == 0)
    except (FileNotFoundError, subprocess.SubprocessError):
        gpu = False
else:
    gpu = device == "gpu"
if not gpu:
    os.environ.setdefault("CUDA_VISIBLE_DEVICES", "-1")
    os.environ.setdefault("JAX_PLATFORMS", "cpu")
tensorflow_version = None
if 'jax' in ("tensorflow", "jax"):
    try:
        tensorflow_version = version("tensorflow")
    except PackageNotFoundError:
        pass
# Colab's CPU image currently carries a CUDA-enabled TensorFlow wheel. Its
# first ordinary tensor operation probes CUDA and emits an error-level cuInit
# diagnostic. JAX notebooks also use TensorFlow for data loading, so overlay
# the matching CPU build in both CPU variants. Keep the provider's
# ``tensorflow`` distribution metadata: other preinstalled Colab packages
# depend on that distribution name, while both wheels expose the same module.
if not gpu and 'jax' in ("tensorflow", "jax"):
    try:
        tensorflow_cpu_version = version("tensorflow-cpu")
    except PackageNotFoundError:
        tensorflow_cpu_version = None
    if (tensorflow_version is not None and
            tensorflow_cpu_version != tensorflow_version):
        subprocess.check_call([
            sys.executable, "-m", "pip", "install", "-q", "--no-deps",
            f"tensorflow-cpu=={tensorflow_version}",
        ])
if "tf-keras" in fallbacks and tensorflow_version is not None:
    fallbacks["tf-keras"] = f"tf-keras=={tensorflow_version}"
missing = []
for package in required:
    if package in pinned:
        wanted, cpu_requirement, gpu_requirement, match = pinned[package]
        requirement = gpu_requirement if gpu else cpu_requirement
        try:
            installed = version(package)
        except PackageNotFoundError:
            installed = None
        actual = (installed.split("+", 1)[0]
                  if installed is not None and match == "public" else installed)
        if actual != wanted:
            missing.append(requirement)
    elif importlib.util.find_spec(imports.get(package, package)) is None:
        missing.append(fallbacks.get(package, package))
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

mismatched = []
for package, (wanted, _, _, match) in pinned.items():
    try:
        installed = version(package)
    except PackageNotFoundError:
        installed = None
    actual = (installed.split("+", 1)[0]
              if installed is not None and match == "public" else installed)
    if actual != wanted:
        mismatched.append(f"{package}={installed!r} (expected {wanted})")
if mismatched:
    raise RuntimeError("Hosted runtime setup failed: " + ", ".join(mismatched))

root = Path(".d2l-hosted") / "5d733eab198ad58f23ceee3f1550014385366ece"
package = root / "d2l"
package.mkdir(parents=True, exist_ok=True)
base = "https://raw.githubusercontent.com/smolix/d2l-neu/5d733eab198ad58f23ceee3f1550014385366ece/d2l"
for name in ('__init__.py', 'jax.py'):
    target = package / name
    if not target.exists():
        urlretrieve(f"{base}/{name}", target)
if str(root.resolve()) not in sys.path:
    sys.path.insert(0, str(root.resolve()))
pythonpath = os.environ.get("PYTHONPATH", "").split(os.pathsep)
if str(root.resolve()) not in pythonpath:
    os.environ["PYTHONPATH"] = os.pathsep.join(
        [str(root.resolve()), *[entry for entry in pythonpath if entry]]
    )


# Multi-GPU in Practice

The hand-rolled loop of that section taught the mechanism and
then lost the race: on a tiny model over a host-staged wire, the second
GPU made things slower. Two things were wrong, and only one was the
hardware. This section fixes the other — the *software* — by replacing our
loop with the production machinery, and in doing so meets the chapter's
sharpest framework contrast: PyTorch makes you launch processes and the
collectives are explicit; JAX runs in one process and you merely *annotate
the layout*, letting the compiler write the collectives for you. We
measure real scaling on 2–4 GPUs, sketch how the same ideas shard a model
too big to replicate (FSDP), and stop at the edge of the single node.

*Prerequisites: the from-scratch data-parallel loop, the ring-allreduce
identity, and the cost model of* that section*; the memory
anatomy of* that section*. The multi-process idiom below
was verified to run under this book's notebook build; why this box's
fabric is slow is the topology story of* that section*,
and what a collective costs on it is* that section*'s
measurement.*

In [ ]:
%matplotlib inline
from d2l import jax as d2l
from flax import nnx
import jax
from jax import numpy as jnp
import numpy as np
import optax
import os
import re
import time

# XLA's NCCL buffer registration fails harmlessly on this P2P-less box;
# opt out rather than let every collective print the warning.
os.environ.setdefault('NCCL_LOCAL_REGISTER', '0')

## What Our Hand-Rolled Loop Lacked

Our that section implementation had three deficits, and modern
data parallelism repairs each.

* **No overlap.** Our loop finished the *entire* backward pass, then called
  `allreduce`. But gradients become available layer by layer, back to
  front, so the last layers' gradients could be communicated while the
  earlier layers are still computing. Serializing compute-then-communicate
  wastes exactly the time the fabric is idle during backward.
* **One Python process.** A single interpreter drove all $k$ GPUs, so one
  GIL-bound thread dispatched every kernel — the overhead regime of
  that section, multiplied by $k$.
* **A star topology.** Our `allreduce` funneled everything through device
  0; that section showed the ring moves a constant per device
  instead.

PyTorch's `DistributedDataParallel` (DDP) fixes all three
[@Li.Zhao.Varma.ea.2020]: one **process per GPU** (no shared GIL),
NCCL's **ring/tree collectives** (no hub), and — the headline —
gradient **bucketing that overlaps communication with the backward pass**
(the figure). As each bucket of gradients fills, DDP kicks
off its allreduce immediately, so by the time the backward pass reaches the
first layer, the last layers' gradients are already summed. This is
compute–communication overlap — independent work scheduled onto the fabric
while the GPUs keep computing — finally shown where it pays.

![What DDP buys over the hand-rolled loop: gradient bucketing lets each
bucket's allreduce overlap the rest of the backward pass, instead of
waiting for all of it.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/mdl-perf-ddp-overlap.svg)

## DDP, Really Run

DDP needs multiple processes, and a notebook is one process — so we launch
the extra ones. The idiom, verified to work under this book's build: write
the training script to a sidecar file, then launch it with `torchrun`,
which spawns one process per GPU, sets up the rendezvous, and runs the
script under `init_process_group`. Each rank writes its result to a scratch
directory the notebook reads back (cleared before every launch, so a
crashed run can never serve stale numbers). We keep the script minimal; the
*loop body* is unchanged from single-GPU training, and the scaffolding
around it is what the launcher's multi-process world requires:

In [ ]:

class ResNet18(nnx.Module):
    """A slightly modified ResNet-18 (small stem, no max-pool)."""
    def __init__(self, num_classes=10, rngs=None):
        rngs = nnx.Rngs(d2l.get_key()) if rngs is None else rngs
        self.net = nnx.Sequential(
            nnx.Conv(1, 64, (3, 3), (1, 1), padding='same', rngs=rngs),
            nnx.BatchNorm(64, rngs=rngs), nnx.relu,
            d2l.Residual(64, in_channels=64, rngs=rngs),
            d2l.Residual(64, in_channels=64, rngs=rngs),
            d2l.Residual(128, use_1x1conv=True, strides=(2, 2),
                         in_channels=64, rngs=rngs),
            d2l.Residual(128, in_channels=128, rngs=rngs),
            d2l.Residual(256, use_1x1conv=True, strides=(2, 2),
                         in_channels=128, rngs=rngs),
            d2l.Residual(256, in_channels=256, rngs=rngs),
            d2l.Residual(512, use_1x1conv=True, strides=(2, 2),
                         in_channels=256, rngs=rngs),
            d2l.Residual(512, in_channels=512, rngs=rngs),
            lambda x: x.mean(axis=(1, 2)),
            nnx.Linear(512, num_classes, rngs=rngs))

    def __call__(self, x):
        return self.net(x)

The DDP training script, written to disk from a cell. The two lines that
make training data-parallel are `init_process_group("nccl")` once per
process and the `DDP(model)` wrap — after those, *the loop body is
unchanged from single-GPU*. The rest is the launcher's housekeeping, shown
in full so none of it is magic: a rank-local device, a
`DistributedSampler` that hands each rank a disjoint shard of the data
(re-shuffled per epoch by `set_epoch`), and process-group teardown. One
practicality earns its own line: the parent notebook downloads
Fashion-MNIST once, quietly, *before* any rank exists, so the ranks never
race on the download:

One word before the numbers, because every published speedup quietly picks
a convention (that section). This sweep holds the
*per-rank* batch fixed at 256, so the global batch grows to $256k$ —
**weak scaling**: the efficiencies it prints are throughput per GPU
relative to one GPU. Remember that a grown global batch also changes the
optimization trajectory itself (that section); the
strong-scaling question — same global batch, finishing sooner — comes
right after.

On our four-GPU box, ResNet-18 on Fashion-MNIST-64 (11.2M parameters)
scales the way the accounting of that section says a
*compute-dense* model should: about 1.8× at two GPUs and about 3.3× at
four — the cell prints 88% and 82% weak-scaling efficiency. The efficiency
sags gently — no cliff — because each step's compute is large enough to
hide most of the communication, unlike LeNet. Strong scaling asks a harder
question of the same hardware: hold the global batch at 512 and split it
ever thinner, so the per-rank batch shrinks as $512/k$ while the allreduce
stays the same size:

At two GPUs the two conventions nearly coincide on this model — a 4090 is
about equally saturated by a per-rank batch of 256 or 512, so the $k=1$
baselines barely differ — but they part company as $k$ grows: by $k=4$ the
strong-scaling run gives each GPU only 128 examples per step, and any gap
that opens between the strong and weak efficiencies is pure
underutilization, the mechanism that sank LeNet, now in miniature. That is
the equation read twice: weak scaling holds $t_{\text{compute}}$
per device constant and asks the fabric to keep up; strong scaling shrinks
$t_{\text{compute}}(B/k)$ toward the fixed $t_{\text{comm}}$ floor.

Now confront the sweep with the cost model by measurement rather than by
assertion. the equation prices a step's communication at
$2N/\beta$, and DDP itself provides the instrument to check it:
`no_sync()` runs backward with gradient synchronization turned off, so the
difference between synced and unsynced step times estimates what
communication really costs — an estimate, not a truth, since turning sync
off also changes what overlaps what:

Prediction and measurement land within tens of percent of each other, and
**that agreement is the result**: a scaling curve you can price before you
buy, not a marketing "N× faster" claim — and it holds at two GPUs as
clearly as at four. One line on what a datacenter box changes: an NVLink
fabric shrinks $t_{\text{comm}}$ by roughly two orders of magnitude
(the table), so the same accounting predicts near-linear
scaling — same model, different constant. (The legacy `nn.DataParallel` is
single-process and GIL-bound; use DDP even on one node, as PyTorch's own
docs advise.)

One loose end from that section deserves to be closed by
measurement rather than left as an assertion. The diagnosis there was
that NCCL's P2P-less fallback moves bytes with a latency-bound GPU-kernel
copy — in effect a performance bug in how the library's default transport
interacts with this particular platform — and that one documented switch,
`NCCL_SHM_USE_CUDA_MEMCPY=1`, re-routes the same transfer over the DMA
copy engines. Measure the bare collective both ways:

Roughly five-fold, from configuration alone. First the general lesson: a
collective library's configuration — which transport it picks, what
topology it assumes — can move communication performance by *factors*,
not percent, so measure yours against what the wire demonstrably carries
(that section's raw copy) before trusting it. Then the
specific one, which is why every training run above still uses the
library's defaults: this workaround wins the microbenchmark and loses
the workload. On our box, with this NCCL build, the copy-engine mode
deadlocks DDP's overlapped training path within seconds — the very
collectives that just ran flawlessly in isolation wedge inside the
training loop — so adopting it here would trade a five-fold bandwidth
win for a hung notebook. An escape hatch is platform-specific twice over:
whether you need it, and whether it survives your workload, are both
measurements (the exercises have you reproduce both halves). The cost
model is indifferent either way — the equation simply takes
whatever $\beta$ your fabric, as configured, sustains.

## Sharding the Redundant: the FSDP Idea

DDP replicates *everything* on every rank: $k$ identical copies of the
parameters, the gradients, and the optimizer states. For the $16P$-byte
training footprint of that section, that is $k-1$ copies
of everything, wasted — and it caps the model size at what one GPU holds,
the limitation that section flagged. **Fully Sharded Data
Parallel** (FSDP) removes the redundancy by *sharding* those tensors across
ranks, each rank owning $1/k$ of each, and materializing a full layer only
for the moment it is needed [@Zhao.Gu.Varma.ea.2023]. The idea is the
ZeRO ladder [@Rajbhandari.Rasley.Ruwase.ea.2020]: shard the
optimizer states first (they are the biggest, $8P$), then the gradients,
then the parameters — each rung cutting memory toward $1/k$ at the cost of
more communication.

The mechanism is the that section identity, cashed in. Recall
that allreduce = reduce-scatter + all-gather. FSDP simply *keeps the two
halves separate*: an **all-gather** reconstructs a layer's full parameters
just before it computes, and frees them just after; a **reduce-scatter**
sums each layer's gradients but leaves each rank holding only its own
shard (the figure). No tensor's full replica ever lives
longer than the layer that needs it.

![The FSDP lifecycle of one block, under one simplified
reshard-after-forward policy: parameters live sharded ($P/k$ per rank); an
all-gather materializes the full block only while it computes, then frees
it; a reduce-scatter leaves each rank with its gradient
shard.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/mdl-perf-fsdp-lifecycle.svg)

That completes the small family of collectives this chapter needs — worth
one table, since the rest of the book will name them without ceremony:

<!-- tbl-caption:The collective operations behind data-parallel and sharded training. -->

| collective | what every rank ends with | where it appears |
|---|---|---|
| allreduce | the full sum | DDP's gradient buckets; that section |
| reduce-scatter | one shard of the sum | FSDP gradients |
| all-gather | every shard, concatenated | FSDP parameters, just-in-time |
| all-to-all | a different shard from each peer | expert parallelism (that section) |

FSDP's payoff — fitting a model that does not fit — is invisible on our
11.2M-parameter demo, which occupies a few hundred MB of a 24 GB card, so
we show the *shape* of the code rather than run it. The modern API is
`fully_shard` over a `DeviceMesh`; the original `FullyShardedDataParallel`
wrapper class still imports at our pin, but it is the deprecated legacy
path:

In [ ]:
# In JAX the same sharding is a PartitionSpec, not a wrapper — see below.
print('JAX shards by annotation; the next subsection is the demo')

You reach for FSDP when the training state at your precision — the
parameters, gradients, and optimizer states of
that section's anatomy, plus activations — no longer
fits on one GPU, or when that redundancy is worth trading away for
communication; for this card class the threshold arrives at a few billion
parameters. The production distributed-training map, and how to combine
FSDP with the other parallelism axes, lives in
that section.

## JAX: Annotate the Layout, the Compiler Writes the Collectives

Everything above was PyTorch's world: multiple processes, explicit
collectives, a launcher. JAX offers a different deal, and it is the
chapter's cleanest framework contrast. One process sees all the GPUs
(that section); you describe *how the data is laid out*
across them with a `Mesh` and a `NamedSharding`, `device_put` the arrays
onto that layout — parameters replicated (`P()`), the batch sharded along
its leading axis (`P('data')`) — and `jit` the **unchanged** single-device
training step. XLA's automatic-parallelization pass (GSPMD
[@Xu.Lee.Chen.ea.2021]) partitions the computation to match the data
layout and *inserts the very allreduce* that that section wrote
by hand — you never write a collective:

In [ ]:
def make_mesh(k):
    return jax.make_mesh((k,), ('data',))

def shard_batch(X, y, mesh):
    P = jax.sharding.PartitionSpec('data')      # leading axis across devices
    sh = jax.sharding.NamedSharding(mesh, P)
    return jax.device_put(X, sh), jax.device_put(y, sh)

def replicate(module, mesh):
    """Give every device of the mesh a full copy of a module's state."""
    sh = jax.sharding.NamedSharding(mesh, jax.sharding.PartitionSpec())
    nnx.update(module, jax.device_put(nnx.state(module), sh))

# The training step is written for one device; jit + sharded inputs make it
# data-parallel with no change to the body.
@nnx.jit
def train_step(model, opt, X, y):
    def loss_fn(m):
        return optax.softmax_cross_entropy_with_integer_labels(
            m(X), y).mean()
    loss, grads = nnx.value_and_grad(loss_fn)(model)
    opt.update(model, grads)                     # XLA inserts the allreduce
    return loss

Now run it — a real `ResNet18`, a real optimizer, a real Fashion-MNIST
batch (resized once to 64×64 and kept as host arrays). The reveal is
`visualize_array_sharding`, which draws where a tensor actually lives: the
batch split across the mesh before the step, and a weight *after* the step
still replicated everywhere — the compiler's allreduce is what kept every
copy identical:

In [ ]:
imgs, labels = d2l.FashionMNIST().train         # raw arrays; resize once
X_all = np.asarray(jax.image.resize(
    jnp.float32(imgs[:, :, :, None]) / 255, (len(imgs), 64, 64, 1),
    'bilinear'))
y_all = np.asarray(labels, np.int32)

k = min(4, jax.local_device_count())
mesh = make_mesh(k)
model = ResNet18(rngs=nnx.Rngs(0))
opt = nnx.Optimizer(model, optax.sgd(0.1), wrt=nnx.Param)
replicate(model, mesh); replicate(opt, mesh)
Xs, ys = shard_batch(X_all[:256], y_all[:256], mesh)
loss = train_step(model, opt, Xs, ys)           # one data-parallel step
print(f'one sharded step: loss {float(loss):.2f}')
print('the batch, split along its leading axis:')
jax.debug.visualize_array_sharding(Xs.reshape(len(Xs), -1))
print('a weight after the step, still replicated:')
jax.debug.visualize_array_sharding(model.net.layers[-1].kernel[...])

Take the receipt. that section wrote its `pmean` by hand; here
nobody wrote a collective — so where is it? In the compiled program. Lower
and compile the step, and search the XLA text for what GSPMD inserted:

In [ ]:
hlo = train_step.lower(model, opt, Xs, ys).compile().as_text()
ops = [line.strip() for line in hlo.splitlines()
       if ' all-reduce(' in line or ' all-reduce-start(' in line]
sizes = [int(re.search(r'\[(\d+)\]', op).group(1)) for op in ops]
print(f'{len(ops)} all-reduce ops in the compiled step; the largest '
      f'sums {max(sizes) / 1e6:.1f}M floats:')
print(ops[sizes.index(max(sizes))][:88] + ' ...')

Dozens of allreduces, not one — and reading them teaches. The two largest
together carry the ~11M gradient floats (XLA bucketed them into two fused
sums, much as DDP buckets); the many small ones are batch-norm statistics,
because under `jit` even normalization is computed over the *global*
batch — the compiler preserves single-device semantics exactly, where DDP
leaves each replica its own batch-norm statistics unless you ask for
`SyncBatchNorm`. (The exact count is a compiler artifact; the two big ones
are the point.)

Measured, under the same weak-scaling convention as the DDP sweep
(per-device batch 256) — one process, no launcher, no sidecar files:

In [ ]:
def batches(global_batch):
    n = (len(X_all) // global_batch) * global_batch
    for i in range(0, n, global_batch):
        yield X_all[i:i + global_batch], y_all[i:i + global_batch]

def jax_throughput(k, per_device_batch=256):
    mesh = make_mesh(k)
    model = ResNet18(rngs=nnx.Rngs(0))
    opt = nnx.Optimizer(model, optax.sgd(0.1), wrt=nnx.Param)
    replicate(model, mesh); replicate(opt, mesh)
    B = per_device_batch * k
    for epoch in range(2):                 # epoch 0 warms up and compiles
        t0 = time.time(); n = 0
        for X, y in batches(B):
            loss = train_step(model, opt, *shard_batch(X, y, mesh))
            n += X.shape[0]
        loss.block_until_ready()           # completion timing
        dt = time.time() - t0
    return n / dt

ks = [k for k in (1, 2, 4) if k <= jax.local_device_count()]
tput = [jax_throughput(k) for k in ks]
for k, t in zip(ks, tput):
    print(f'{k} GPU(s): {t:.0f} samples/s, {t / tput[0]:.2f}x, '
          f'{100 * t / tput[0] / k:.0f}% weak-scaling efficiency')

The shape matches the DDP sweep — real gains, efficiency sagging as $k$
grows, no cliff — with one instructive difference: the compiled XLA step
is faster per device, so the same allreduce is a *larger fraction* of each
step and the efficiency reads a few points lower (82% already at two GPUs
in our runs). That is the equation again: speed up the compute and
the fabric gets *relatively* slower. And note what these numbers are not:
this loop feeds pre-staged host arrays while the DDP script pays a real
`DataLoader`, so the absolute samples/s are not a framework shoot-out —
the scaling curve is the comparison.

One more check, because we are trusting a collective nobody wrote.
that section ended its hand-built loop with a one-step equality
test, and the declarative version deserves the same scrutiny — same
initialization, same 256-example batch, one step on one device versus one
step sharded across two:

In [ ]:
def one_step_delta(k):
    mesh = make_mesh(k)
    model = ResNet18(rngs=nnx.Rngs(0))
    opt = nnx.Optimizer(model, optax.sgd(0.1), wrt=nnx.Param)
    replicate(model, mesh); replicate(opt, mesh)
    before = jax.tree.map(np.asarray, nnx.state(model, nnx.Param))
    train_step(model, opt, *shard_batch(X_all[:256], y_all[:256], mesh))
    after = jax.tree.map(np.asarray, nnx.state(model, nnx.Param))
    return jax.tree.map(lambda a, b: a - b, after, before)

d1 = one_step_delta(1)
d2 = one_step_delta(min(2, jax.local_device_count()))
gaps = jax.tree.map(lambda a, b: np.abs(a - b).max(), d1, d2)
print('max update difference, k=1 vs k=2: '
      f'{max(jax.tree.leaves(gaps)):.1e}')

The two runs agree to about $10^{-4}$ against update magnitudes near
$10^{-2}$ — not the $10^{-9}$ of that section's LeNet check,
and the gap is itself informative: these are two *different compiled
programs*, whose tf32 convolutions and batch-norm reductions associate
their arithmetic differently, so the residue is rounding. What the check
exists to catch — a summed-instead-of-averaged gradient, an accidental
$k\times$ learning rate — would announce itself at the scale of the update
itself.

And here is the punchline. To move between *data* parallelism, *tensor*
parallelism, and *FSDP*-style sharding in JAX, you change the
`PartitionSpec` — not the model code. Sharding the batch axis gives data
parallelism (above); sharding a weight's feature axis gives tensor
parallelism; sharding the parameters and letting XLA all-gather them
just-in-time gives the FSDP pattern. **One sharding vocabulary — annotate
the layout, the compiler writes the collectives — spans what PyTorch
exposes as three different APIs** (DDP, tensor-parallel wrappers, FSDP) —
though an effective sharding plan is still model-aware: someone has to
choose the layouts and the constraint points. The manual end of the same
spectrum is the `jax.shard_map` + `lax.psum` of that section,
where you write the collective yourself; `jit` + sharding is the
automatic end.

<!-- tbl-caption:Multi-GPU parallelism in PyTorch versus JAX. -->

| | PyTorch | JAX |
|---|---|---|
| processes | one per GPU (`torchrun`) | one, sees all GPUs |
| collectives | explicit (NCCL, or DDP's buckets) | inserted by XLA/GSPMD |
| data parallel | `DistributedDataParallel` | shard the batch axis |
| tensor parallel | separate wrappers / DTensor | change the `PartitionSpec` |
| sharded (FSDP) | `fully_shard` | change the `PartitionSpec` |
| control | imperative, visible | declarative, compiler-driven |

Neither deal is strictly better: PyTorch's explicitness makes the
communication legible and debuggable; JAX's declarativeness makes the same
code span parallelism strategies by editing an annotation. Knowing both is
knowing the design space.

## When One Node Is Not Enough

This chapter stops at the boundary of a single machine, and it is worth
naming what lies past it. When a model is too large for even sharded data
parallelism on one node — when the parameters, or the batch, or the
sequence length outgrow what $k$ local GPUs can hold — the answer is to
shard across *machines*, combining data parallelism with tensor, pipeline,
and expert parallelism into the "3D parallelism" of frontier training. The
collectives then run over a network fabric measured in tens of GB/s
between nodes rather than an NVLink domain within one, and the cost model
of that section acquires a second, slower bandwidth term. That
is the province of the Language Models part, which has models and datasets
large enough to warrant it; the production library map — Megatron, the
FSDP/DTensor stack, DeepSpeed, and how to launch and checkpoint them across
a cluster — is that section. From here the communication
*algebra* stays the same; what multi-node adds is engineering on top of
it — a hierarchy of fabrics (NVLink inside a node, a network between
nodes), rendezvous and elastic restart when machines fail, and stragglers
that turn a synchronous step into a queueing problem.

## Summary

* Production data parallelism (DDP) fixes the three deficits of a
  hand-rolled loop: one process per GPU (no shared GIL), ring/tree
  collectives (no hub), and gradient bucketing that **overlaps
  communication with the backward pass**.
* Launched from a notebook via `torchrun` on a sidecar script, DDP scales
  a compute-dense ResNet-18 by roughly 1.8× on two GPUs and 3.3× on four
  on our host-staged box (88% and 82% *weak-scaling* efficiency; a
  fixed-global-batch strong-scaling sweep asks the harder question), and a
  `no_sync()` measurement confirms the equation's communication
  price to within tens of percent. An NVLink box changes only the
  constant.
* The constant is also configuration-sensitive: one documented NCCL
  switch quintuples the bare-collective bandwidth on this box — yet
  deadlocks DDP's overlapped training path, so the notebooks keep the
  library's defaults and teach the pair of measurements instead.
  Collective-library configuration moves communication by factors, not
  percent; a workaround is validated per platform *and* per workload.
* FSDP shards the $16P$-byte training state across ranks — the ZeRO ladder
  — by splitting allreduce back into its reduce-scatter and all-gather
  halves and materializing each layer just-in-time. It is for models whose
  training state outgrows one GPU — a few billion parameters on this card
  class.
* JAX inverts the model: one process, annotate the data layout with a
  `Mesh` and `PartitionSpec`, `jit` the unchanged step, and XLA inserts
  the collectives — visible in the compiled program's `all-reduce` ops,
  measured in a k-sweep, and verified by a k=1-versus-k=2 equality check.
  Changing the `PartitionSpec` — not the code — moves between data,
  tensor, and FSDP-style sharding, spanning what PyTorch exposes as three
  APIs.
* Multi-node 3D parallelism and the production library map are the
  Language Models part and that section.

## Exercises

1. Vary DDP's `bucket_cap_mb` (the gradient bucket size) and measure
   throughput at $k = 2$. Why is there an optimum — what does a too-small
   bucket cost, and a too-large one?
1. Extend the `no_sync()` measurement to $k = 4$. How much does the
   per-step communication time grow from $k=2$? Compare against
   the equation's $2(k-1)/k$ growth and the flat $2N/\beta$
   of the equation — which fits better, and what does that tell
   you about how NCCL schedules the transfer on this P2P-less box?
1. Reproduce both halves of the fabric-configuration story. First extend
   the bare-collective comparison across payloads from 1 MB to 256 MB:
   where does each transport's effective bandwidth saturate, and does
   the five-fold gap persist? Then set `NCCL_SHM_USE_CUDA_MEMCPY=1`
   inside `train_ddp.py` itself and relaunch the $k=2$ sweep — on our
   box the run wedges within seconds (be ready to kill the launcher).
   What does the pair teach about trusting a library's defaults on a
   topology it was not tuned for — and about trusting a workaround
   anywhere you have not re-validated it, workload and all?
1. Write the `PartitionSpec` that shards a weight matrix's *output*
   features across the mesh (tensor parallelism) and
   `visualize_array_sharding` the result. How does the communication
   pattern differ from the batch-sharded (data-parallel) case?
1. Size ZeRO stage 3 (parameters, gradients, and optimizer states all
   sharded) for a 7-billion-parameter model on 8 GPUs with 80 GB each:
   what is the per-GPU training footprint, and does it fit? Redo the
   arithmetic for DDP (no sharding) and explain the difference.
1. The weak-scaling efficiency in the DDP sweep fell from 88% at $k=2$ to
   82% at $k=4$. Using the equation, predict the efficiency at
   $k=8$ on the same fabric, and state the assumption your prediction
   makes about how $t_{\text{comm}}$ grows.